In [1]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset(
    "csv", # csv format
    data_files="processed_reviews_2.csv", # path and name for the file
    encoding="latin-1" # latin-1 encoding to avoid character errors
)

# Format the dataset
def format_example(example):
    return {
        "text": f"Game: {example['app_name']}\nInput: {example['input']}\nOutput: {example['output']}" # formating used
    }

dataset = dataset.map(format_example) #maps the format

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/315 [00:00<?, ? examples/s]

In [2]:
# Load the base model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Load model weights in 4-bit precision
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computations
    bnb_4bit_use_double_quant=True, # Enable double quantization for better compression
    bnb_4bit_quant_type="nf4" # Use NF4 (normalized float 4) quantization type
)

#from huggingface_hub import login
#login(token="HF_TOKEN") # for a faster download 14.5 GB

model_name = "mistralai/Mistral-7B-v0.1" # specifies the base model

tokenizer = AutoTokenizer.from_pretrained(model_name) # loads tokenizer

# padding
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example): # Tokenization function: converts text into numerical tokens the model can understand
    tokens = tokenizer(
        example["text"], # Input text to tokenize
        truncation=True, # Truncate text if it exceeds max length
        padding="max_length", # Pad all sequences to the same length
        max_length=256 # Maximum sequence length
    )
    
    # Labels are the same as input_ids for causal language modeling
    tokens["labels"] = tokens["input_ids"].copy()
    
    return tokens

dataset = dataset.map(tokenize, batched=True) # Apply the tokenization

dataset = dataset["train"].train_test_split(test_size=0.1) # Split train data 90%, test data 10%

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Apply 4-bit quantization settings
    device_map="auto" # Automatically place model on available hardware
)

Map:   0%|          | 0/315 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
# Dataset for rouge evaluation
eval_dataset = load_dataset(
    "csv",
    data_files="processed_reviews_2.csv",
    encoding="latin-1"
)["train"]

eval_dataset = eval_dataset.train_test_split(test_size=0.1)["test"] # same test/train split

In [4]:
import evaluate

rouge = evaluate.load("rouge") # Load ROUGE metric for comparing generated text vs reference text

predictions = [] # Store model-generated outputs
references = [] # Store ground truth outputs

model.eval() # Set model to evaluation mode (disables dropout, etc.)

for example in eval_dataset: # Loop through evaluation dataset
    prompt = f"Game: {example['app_name']}\nInput: {example['input']}\nOutput:" # Create prompt WITHOUT the expected output (model must generate it)
    
    inputs = tokenizer( # Tokenize the prompt and move it to the same device as the model (GPU/CPU)
        prompt,
        return_tensors="pt", # Return PyTorch tensors
        truncation=True, # Truncate if too long
        max_length=256 # Max input length
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100, # Limit length of generated response
        do_sample=False # Disable sampling (greedy decoding)
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True) # Convert generated tokens back into readable text
    
    prediction = generated_text.split("Output:", 1)[-1].strip() # Extract only the generated "Output" portion
    
    # Save prediction and corresponding reference answer
    predictions.append(prediction)
    references.append(example["output"])

base_results = rouge.compute( # Compute ROUGE scores to evaluate model performance
    predictions=predictions,
    references=references
)

print(base_results)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

{'rouge1': np.float64(0.22672504526875922), 'rouge2': np.float64(0.05307899320634558), 'rougeL': np.float64(0.13338479773514123), 'rougeLsum': np.float64(0.16073099617261327)}


In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig( # Apply LoRA adding trainable adapters
    r=16, # Rank of the low-rank matrices (controls adapter size)
    lora_alpha=32, # Scaling factor for LoRA updates
    target_modules=["q_proj", "v_proj"],  # Apply LoRA to attention projection layers
    lora_dropout=0.05, # Dropout for regularization (helps prevent overfitting)
    bias="none", # Do not update bias parameters
    task_type="CAUSAL_LM" # Specify task type (causal language modeling)
)

model = get_peft_model(model, lora_config) # Wrap the base model with LoRA adapters (only small parts become trainable)
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [6]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments( # Setup trainer
    output_dir="./7b_lora_model", # Directory to save model checkpoints and outputs
    per_device_train_batch_size=1, # Batch size per GPU (kept small due to large model size)
    gradient_accumulation_steps=8, # Accumulate gradients to simulate batch size of 8
    learning_rate=2e-4, # Learning rate for optimizer
    fp16=True, # Enable mixed precision (faster + less memory)
    save_strategy="epoch", # Save model at the end of each epoch
    logging_steps=10, # Log training progress every 10 steps
    max_steps=-1, # Use epochs instead of a fixed number of steps
    num_train_epochs=3, # Train for 3 full passes over the dataset
    optim="paged_adamw_32bit", # Memory-efficient optimizer for large models
)

trainer = Trainer(
    model=model, # Model with LoRA applied
    args=training_args, # Training configuration
    train_dataset=dataset["train"], # Training dataset (90%)
    eval_dataset=dataset["test"] # Evaluation dataset (10%)
)

In [7]:
trainer.train() # Train on the data

Step,Training Loss
10,2.319685
20,2.068013
30,2.127330
40,2.066042
50,1.953486
60,1.972986
70,1.957106
80,1.897779
90,1.874880
100,1.868307


TrainOutput(global_step=108, training_loss=1.9990561979788322, metrics={'train_runtime': 734.2595, 'train_samples_per_second': 1.156, 'train_steps_per_second': 0.147, 'total_flos': 9281644013813760.0, 'train_loss': 1.9990561979788322, 'epoch': 3.0})

In [8]:
# save the model
model.save_pretrained("lora_model_7B")
tokenizer.save_pretrained("lora_model_7B")

('lora_model_7B\\tokenizer_config.json', 'lora_model_7B\\tokenizer.json')

In [3]:
# reload the model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "lora_model_7B")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
# Dataset for rouge evaluation
from datasets import load_dataset

eval_dataset = load_dataset(
    "csv",
    data_files="processed_reviews_2.csv",
    encoding="latin-1"
)["train"]

eval_dataset = eval_dataset.train_test_split(test_size=0.1)["test"] # same test/train split

In [6]:
# Rouge eval for trained model
import evaluate

rouge = evaluate.load("rouge")

predictions = []
references = []

model.eval()

for example in eval_dataset:
    prompt = f"Game: {example['app_name']}\nInput: {example['input']}\nOutput:"
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    prediction = generated_text.split("Output:", 1)[-1].strip()
    
    predictions.append(prediction)
    references.append(example["output"])

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

{'rouge1': np.float64(0.23162633071413669), 'rouge2': np.float64(0.06602154665347074), 'rougeL': np.float64(0.14666026094530027), 'rougeLsum': np.float64(0.16709832223508236)}
